### 🏷️ Credits & License
- 🔗 OmniVoice GitHub Repository
- 🤗 OmniVoice on Hugging Face
- 📄 Apache 2.0 License

### ⚠️ Disclaimer
Use only for ethical voice cloning purposes. No impersonation or fraud.

In [ ]:
#@title INSTALL DEPENDENCIES
%cd /content/

!rm -rf omnivoice-colab
!git clone https://github.com/hahyt6644-gif/omnivoice-colab.git
%cd /content/omnivoice-colab

!rm -rf OmniVoice
!git clone https://github.com/NeuralFalconYT/OmniVoice.git

!apt-get update -qq
!apt-get install -y ffmpeg

!pip install -q -r colab.txt
!pip install -q fastapi uvicorn[standard] python-multipart httpx requests aiofiles nest-asyncio pydub soundfile librosa

# cloudflared
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

import torch, sys
sys.path.append('/content/omnivoice-colab/OmniVoice')

print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')
print('INSTALL DONE')


In [ ]:
#@title START SERVER + STABLE TUNNEL

import subprocess, time, requests, re

%cd /content/omnivoice-colab

# kill old processes
!pkill -f app.py || true
!pkill -f cloudflared || true

print('🚀 Starting OmniVoice Server...')

# start API server
server = subprocess.Popen(['python', 'app.py'])

# IMPORTANT: wait for full model load
time.sleep(25)

print('🌐 Starting Cloudflare Tunnel...')

# start tunnel
tunnel = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://localhost:7860'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

url = None

print('⏳ Waiting for public URL...')

for _ in range(150):
    line = tunnel.stdout.readline().strip()

    if 'trycloudflare.com' in line:
        m = re.search(r'https://[-a-zA-Z0-9]+\.trycloudflare\.com', line)
        if m:
            url = m.group(0)
            break

    time.sleep(0.3)

if url:
    print('\n' + '='*60)
    print('✅ PUBLIC URL READY')
    print(url)
    print('\nUI:', url + '/ui')
    print('API:', url + '/api/clone')
    print('='*60)
else:
    print('❌ Failed to generate URL - retry cell')
